# ETL & Assumptions Clinic

**DS2002 | 2026-04-13 | Studio (Hands-On Lab)**

---

## Overview

Today is a working session. Your goals:

1. Finish your data cleaning pipeline.
2. Consolidate vehicle IDs (the Pop-Tarts problem).
3. Load cleaned data into a SQLite database.
4. Upload your cleaned files to your GCS team folder.
5. Start pulling external API data.

Your instructor and TAs are available to troubleshoot. Bring your questions.

## Part 1 — Cleaning the session data

The `charging_sessions.csv` has the most issues. Here is a suggested cleaning order:

1. **Drop exact duplicates** — `df.drop_duplicates()`
2. **Standardize station IDs** — map all variants (e.g., `STN001`, `STN_007`, `STN-007`) to the canonical format (`STN-XXX`)
3. **Parse dates** — the `session_start` and `session_end` columns use at least 5 different formats. Use `pd.to_datetime()` with multiple passes or a custom parser.
4. **Fix cost column** — strip `$` symbols, convert to float
5. **Fix kWh column** — remove negative values (data entry errors) or take absolute value
6. **Handle missing values** — replace `"NULL"`, `"N/A"`, `"NaN"`, `"None"`, `"nan"` with actual `NaN`, then decide: drop or impute

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import os

sessions = pd.read_csv("data/charging_sessions.csv")
print(f"Raw sessions: {sessions.shape}")
sessions.head()

In [ ]:
# Step 1: Drop exact duplicates
before = len(sessions)
sessions = sessions.drop_duplicates()
print(f"Dropped {before - len(sessions)} exact duplicates. Remaining: {len(sessions)}")

In [ ]:
# Step 2: Standardize station IDs
# Find all unique station_id values to understand the variants
print("Unique station IDs:")
print(sorted(sessions["station_id"].unique()))

In [ ]:
# Step 3: Parse dates — build a multi-format parser
# Your code here


In [ ]:
# Step 4: Fix cost column — strip $ and convert
# Your code here


In [ ]:
# Step 5: Fix kWh — handle negatives
# Your code here


In [ ]:
# Step 6: Handle missing values
# Replace string nulls with actual NaN
null_strings = ["NULL", "N/A", "NaN", "nan", "None", ""]
# Your code here


## Part 2 — Vehicle ID consolidation

This is the core data challenge. The same vehicle appears under multiple IDs.

Approach:
1. Look at `vehicle_types.csv` — it has the messy ID-to-name mappings.
2. Build a mapping dictionary: every variant ID maps to a canonical vehicle name.
3. Apply the mapping to the session data.
4. Compare before vs. after.

In [ ]:
vehicles = pd.read_csv("data/vehicle_types.csv")
print(f"Vehicle reference: {vehicles.shape}")
vehicles

In [ ]:
# Build your vehicle ID mapping
# vehicle_map = { "VH-001": "Tesla Model 3", "VEH#0001": "Tesla Model 3", ... }
# Your code here


In [ ]:
# Apply mapping to sessions
# sessions["vehicle_name"] = sessions["vehicle_id"].map(vehicle_map)
# Your code here


## Part 3 — Load into SQLite

Create a local SQLite database with your cleaned tables.

In [ ]:
# conn = sqlite3.connect("ev_analytics.db")
# sessions_clean.to_sql("sessions", conn, if_exists="replace", index=False)
# stations_clean.to_sql("stations", conn, if_exists="replace", index=False)
# conn.close()
# Your code here


## Part 4 — Upload to GCS

Upload your cleaned CSV and SQLite database to your team folder.

In [ ]:
# Authenticate (same pattern as the April 1 lab)
# from google.colab import auth
# auth.authenticate_user()
# from google.cloud import storage
# client = storage.Client(project="ds2002-492012")
# bucket = client.bucket("ds2002-capstone-sp26-v2")
#
# TEAM = "team-XX"
# for fname in ["cleaned_sessions.csv", "ev_analytics.db"]:
#     blob = bucket.blob(f"{TEAM}/{fname}")
#     blob.upload_from_filename(fname)
#     print(f"Uploaded {fname}")


## Part 5 — External API setup

Start pulling your weather or energy API data. At minimum, test the API call and parse the response into a DataFrame.

Here is a starter for Open-Meteo (historical weather for Charlottesville):

In [ ]:
import requests

# Charlottesville coordinates: 38.03, -78.48
# Example: get daily temperature for January 2025
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 38.03,
    "longitude": -78.48,
    "start_date": "2025-01-01",
    "end_date": "2025-01-31",
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
    "timezone": "America/New_York"
}

resp = requests.get(url, params=params)
data = resp.json()

weather = pd.DataFrame({
    "date": data["daily"]["time"],
    "temp_max_f": [c * 9/5 + 32 for c in data["daily"]["temperature_2m_max"]],
    "temp_min_f": [c * 9/5 + 32 for c in data["daily"]["temperature_2m_min"]],
    "precip_mm": data["daily"]["precipitation_sum"]
})

print(f"Weather data: {weather.shape}")
weather.head()

---

## End of session checklist

- [ ] Session data cleaned (duplicates, dates, costs, kWh, nulls)
- [ ] Vehicle IDs consolidated
- [ ] Cleaned data loaded into SQLite
- [ ] Cleaned files uploaded to GCS team folder
- [ ] API call tested and returning data

Anything not done today, finish before the Analysis Clinic on Tuesday.